# 03 - Anomaly Detection (Phase 1: z-score baseline)

Step 4/5 of the Karpathy method: visualize, then test, before adding complexity.
This notebook only uses a per-driver z-score on lap time - the simplest possible SCADA-style alarm rule.

In [1]:
import sys
sys.path.insert(0, "..")

from src.data_cleaning import load_and_clean_all
from src.anomaly_detection import add_lap_time_zscore, get_anomaly_table

laps, weather, telemetry = load_and_clean_all()
scored = add_lap_time_zscore(laps)
scored[["Driver", "LapNumber", "LapTimeSeconds", "LapTimeZScore"]].head()

,Driver,LapNumber,LapTimeSeconds,LapTimeZScore
0,ALB,2.0,98.826,0.100067
1,ALB,3.0,98.507,0.022690
2,ALB,4.0,98.422,0.002073
3,ALB,5.0,98.509,0.023176
4,ALB,6.0,98.575,0.039184


In [2]:
anomaly_table = get_anomaly_table(laps)
anomaly_table

,Driver,LapNumber,LapTimeSeconds,Compound,TyreLife,LapTimeZScore
122,BOT,13.0,121.081,HARD,1.0,6.678380
276,HUL,2.0,130.911,HARD,1.0,5.399448
231,HAM,13.0,118.645,HARD,1.0,5.305074
397,MAG,12.0,119.682,HARD,1.0,5.230223
1060,ZHO,10.0,119.328,HARD,1.0,5.173212
954,TSU,15.0,119.588,HARD,1.0,5.171206
564,PER,13.0,117.624,HARD,1.0,5.125203
730,RUS,12.0,118.460,HARD,1.0,5.112846
789,SAI,15.0,117.225,HARD,1.0,5.103831
14,ALB,16.0,119.341,HARD,1.0,5.076188


In [3]:
import plotly.express as px

fig = px.scatter(
    scored,
    x="LapNumber",
    y="LapTimeSeconds",
    color=scored["LapTimeZScore"].abs() > 2.5,
    facet_col=None,
    title="Lap times with anomaly flag (|z| > 2.5)",
    labels={"color": "IsAnomaly"},
)
fig

## What this catches (and what it misses)

Most flagged laps are first-lap-on-a-new-compound or safety-car-affected laps - real outliers, but not interesting *failures*. This is the expected limitation of a static z-score baseline: it has no notion of context (pit stops, track status, weather). Phase 2 should condition the anomaly score on stint/lap context rather than treating every lap as i.i.d.